# Diffusion (DDPM) 이미지 생성 — Digit Recognizer (PyTorch) — Colab

**DDPM**(Denoising Diffusion Probabilistic Model)을 **처음부터 구현**해 MNIST 손글씨 숫자를 생성하는 기본 예제입니다.

- 핵심 흐름: 이미지에 **노이즈를 점진적으로 추가**(forward) → 신경망이 **노이즈를 예측**해 거꾸로 제거(reverse).
- IOAI 2024 CV/생성 과제(Diffusion 모델)의 기반 기법입니다. (우리 14번 DCGAN 대비 최신 생성기법)
- [Digit Recognizer](https://www.kaggle.com/c/digit-recognizer) 데이터를 사용하며 토큰이 **노트북에 내장**되어 바로 실행됩니다.
- 생성 과제라 제출 리더보드는 없습니다(생성 이미지 시각화·저장).

> ⚙️ **GPU 권장**: 런타임 → 런타임 유형 변경 → GPU.  
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Digit Recognizer 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("digit-recognizer", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 데이터 준비

픽셀을 `[-1, 1]` 로 정규화하고 28×28 → **32×32 패딩**(깔끔한 다운샘플링용).

In [ ]:
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

train = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
X = train.drop("label", axis=1).values.reshape(-1, 1, 28, 28).astype("float32") / 255.0
X = X * 2 - 1  # [-1, 1]
X = np.pad(X, ((0,0),(0,0),(2,2),(2,2)), mode="constant", constant_values=-1)  # 32x32
X = torch.from_numpy(X)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loader = DataLoader(TensorDataset(X), batch_size=128, shuffle=True, drop_last=True)
print("device:", device, "| data:", tuple(X.shape))

## 3. 노이즈 스케줄 (DDPM forward process)

선형 베타 스케줄로 `alpha_bar` 를 미리 계산합니다.

In [ ]:
T = 300  # diffusion steps
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

def q_sample(x0, t, noise):
    """x0 에 t 단계 노이즈를 한 번에 추가"""
    ab = alpha_bars[t].view(-1, 1, 1, 1)
    return torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise
print("noise schedule ready | T =", T)

## 4. 노이즈 예측 U-Net (시간 임베딩 포함)

In [ ]:
import torch.nn as nn, math

def time_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class Block(nn.Module):
    def __init__(self, ci, co, tdim):
        super().__init__()
        self.conv1 = nn.Conv2d(ci, co, 3, padding=1); self.conv2 = nn.Conv2d(co, co, 3, padding=1)
        self.temb = nn.Linear(tdim, co); self.act = nn.SiLU(); self.bn = nn.BatchNorm2d(co)
    def forward(self, x, t):
        h = self.act(self.bn(self.conv1(x)))
        h = h + self.temb(t)[:, :, None, None]
        return self.act(self.conv2(h))

class UNet(nn.Module):
    def __init__(self, tdim=128):
        super().__init__()
        self.tdim = tdim
        self.tmlp = nn.Sequential(nn.Linear(tdim, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.d1 = Block(1, 64, tdim); self.d2 = Block(64, 128, tdim)
        self.pool = nn.MaxPool2d(2); self.bott = Block(128, 256, tdim)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = Block(256, 128, tdim)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2); self.c1 = Block(128, 64, tdim)
        self.out = nn.Conv2d(64, 1, 1)
    def forward(self, x, t):
        temb = self.tmlp(time_embedding(t, self.tdim))
        x1 = self.d1(x, temb)               # 32
        x2 = self.d2(self.pool(x1), temb)   # 16
        b = self.bott(self.pool(x2), temb)  # 8
        h = self.c2(torch.cat([self.up2(b), x2], 1), temb)   # 16
        h = self.c1(torch.cat([self.up1(h), x1], 1), temb)   # 32
        return self.out(h)

model = UNet().to(device)
print("params:", sum(p.numel() for p in model.parameters()))

## 5. 학습 (노이즈 예측 MSE)

랜덤 단계 `t` 의 노이즈를 추가하고, 모델이 그 노이즈를 예측하도록 학습합니다.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=2e-4)

EPOCHS = 20  # Colab GPU 기준. 늘릴수록 품질 향상
for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0; n = 0
    for (x0,) in loader:
        x0 = x0.to(device); bs = x0.size(0)
        t = torch.randint(0, T, (bs,), device=device)
        noise = torch.randn_like(x0)
        xt = q_sample(x0, t, noise)
        pred = model(xt, t)
        loss = F.mse_loss(pred, noise)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item() * bs; n += bs
    print(f"epoch {epoch:2d} | noise MSE = {running/n:.4f}")

## 6. 샘플링 (reverse process) & 시각화

순수 노이즈에서 시작해 t=T-1→0 으로 한 단계씩 노이즈를 제거합니다.

In [ ]:
@torch.no_grad()
def sample(n_img=64):
    model.eval()
    x = torch.randn(n_img, 1, 32, 32, device=device)
    for ti in reversed(range(T)):
        t = torch.full((n_img,), ti, device=device, dtype=torch.long)
        eps = model(x, t)
        a = alphas[ti]; ab = alpha_bars[ti]; beta = betas[ti]
        coef = (1 - a) / torch.sqrt(1 - ab)
        mean = (x - coef * eps) / torch.sqrt(a)
        x = mean + (torch.sqrt(beta) * torch.randn_like(x) if ti > 0 else 0)
    return x.clamp(-1, 1)

import matplotlib.pyplot as plt
from torchvision.utils import make_grid
gen = sample(64).cpu()
gen = gen[:, :, 2:30, 2:30]  # 32->28 중앙 크롭
grid = make_grid(gen, nrow=8, normalize=True)
plt.figure(figsize=(8, 8)); plt.imshow(grid.permute(1, 2, 0)); plt.axis("off"); plt.title("DDPM Generated Digits"); plt.show()

## 7. 생성 이미지 저장 & 내려받기

In [ ]:
from torchvision.utils import save_image
import zipfile, glob
OUT = os.path.join(WORK_DIR, "ddpm_samples"); os.makedirs(OUT, exist_ok=True)
for i, img in enumerate((gen + 1) / 2):
    save_image(img, os.path.join(OUT, f"{i:03d}.png"))
zip_path = os.path.join(WORK_DIR, "ddpm_samples.zip")
with zipfile.ZipFile(zip_path, "w") as zf:
    for p in glob.glob(os.path.join(OUT, "*.png")): zf.write(p, os.path.basename(p))
print("Saved:", zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print("자동 다운로드 불가:", e, "| 위치:", zip_path)

## 🏁 제출 안내

이 노트북은 **이미지 생성(Diffusion)** 데모라 제출 리더보드가 없습니다.

- 데이터 출처: **[Digit Recognizer](https://www.kaggle.com/c/digit-recognizer)**
- 생성 결과는 `ddpm_samples.zip` 으로 저장·다운로드됩니다.

> IOAI 2024 CV/생성 과제(SDXL 등 Diffusion 모델 조작)의 기반이 되는 기법입니다.